# CML3111 Engineering ML project: Part A 30 + Part B 15 marks, scaled to 40 percent
1. Run **Setup** (the next cell) first. **Do not edit it.** It shows your time left, and the time shows again after every cell.
2. Do the tasks in order. Write your own code under each task and add cells as you need. Store each result under the name given.
3. There is no Submit button. Your work is collected at your deadline. Press Ctrl+S now and then.
No AI tools and no help from other people. You may be asked to explain your code.
*Data: Ames, Iowa house sales 2006 to 2010 (De Cock 2011); AI4I 2020 machine data, which is synthetic (Matzka 2020, CC BY 4.0).*

In [ ]:
#@title Setup: run me first. DO NOT EDIT THIS CELL
# IUEA-SETUP-CELL (supplied). DO NOT EDIT OR DELETE THIS CELL. Run it first, and again after a restart. It shows your time left.
# Any change made here is undone: the marking computer always uses the original of this cell.
_IUEA_IDENTITY = {"course": "CML3111", "test": "ML_PROJECT"}
import base64, bz2, hashlib, io, json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import confusion_matrix, precision_score, recall_score
FULL_NAME = _IUEA_IDENTITY.get("name", "")
STUDENT_ID = _IUEA_IDENTITY.get("registration", "")
COHORT = _IUEA_IDENTITY.get("course", "")
TEST_ID = _IUEA_IDENTITY.get("test", "")
_IUEA_ALLOCATION = _IUEA_IDENTITY.get("allocation", "")
_IUEA_DEADLINE = _IUEA_IDENTITY.get("deadline", "")
_PACKED = '''
...
'''
_bytes = bz2.decompress(base64.b64decode("".join(_PACKED.split())))
if hashlib.sha256(_bytes).hexdigest() != 'ba6a206588f622cfb58611566c70cdb059f2d21df217fe71d4d1e7eb21d97baf':
    raise RuntimeError("The data in this cell is damaged. Tell your lecturer.")
_data = json.loads(_bytes)
# Houses: Ames, Iowa sales 2006 to 2010 (De Cock 2011). Machines: AI4I 2020, synthetic data modelled on real machine logs (Matzka 2020).
# Both tables were organised for this project, and a few problems were ADDED on purpose for you to clean (see "The data").
H_TARGET, H_UNIT = 'sale_price_usd', 'USD'
H_FEATURES = ['overall_quality', 'overall_condition', 'year_built', 'year_remodelled', 'living_area_m2', 'basement_area_m2', 'lot_area_m2', 'lot_frontage_m', 'bedrooms', 'full_bathrooms', 'garage_cars', 'garage_area_m2', 'fireplaces']
TEXT_FEATURES = ['neighbourhood', 'building_type', 'kitchen_quality']
M_TARGET, M_LIMIT = 'failure', 1
M_FEATURES = ['air_temp_k', 'process_temp_k', 'rotation_speed_rpm', 'torque_nm', 'tool_wear_min', 'product_type']
THRESHOLD = 0.5
MAX_MAE, MAX_MS = 25000, 50   # the client's limits: teaching numbers, not real ones
LATENCY_MS = {'baseline': 1, 'linear': 3, 'tree': 30}
houses_raw = pd.read_csv(io.StringIO(_data["houses"]))
machines_raw = pd.read_csv(io.StringIO(_data["machines"]))
def choose_model(maes):
    ok = [m for m in maes if maes[m] is not None and maes[m] <= MAX_MAE and LATENCY_MS[m] <= MAX_MS]
    return min(ok, key=lambda m: (maes[m], LATENCY_MS[m], m)) if ok else None
print("houses_raw:", houses_raw.shape, "| machines_raw:", machines_raw.shape)

# Automatic backup (supplied). Inside Colab, a copy of the notebook on your screen goes to your lecturer every 5 minutes.
# You do not need to do anything. Your work is collected automatically at your deadline.
def _iuea_backup_start(address='https://iuea-test-entry.jothammartin-wambi.workers.dev/api/backup', minutes=5):
    import json, threading, time, urllib.request
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or "google.colab" not in type(shell).__module__:
            return                                   # not in Colab (the marking computer, Check my notebook): no backup
        from google.colab import _message, output
        from IPython.display import display, HTML
    except Exception:
        return
    attempt = str(globals().get("_IUEA_ALLOCATION", ""))
    if not attempt:
        print("Backup is off: this notebook has no attempt code. Open it from the entry page (Open my project in Colab).")
        return
    old = globals().get("_iuea_backup_thread")
    if old is not None and old.is_alive():
        return
    note = display(HTML(""), display_id=True)
    packed = _PACKED.strip()
    def send():
        notebook = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
        for cell in notebook.get("cells", []):
            text = "".join(cell.get("source", "")) if isinstance(cell.get("source"), list) else str(cell.get("source", ""))
            if "# IUEA-SETUP-CELL" in text:
                cell["source"] = text.replace(packed, "__IUEA_" + "PACKED_DATA__")
        href = str(output.eval_js("location.href"))[:500]
        body = json.dumps({"attempt": attempt, "href": href, "notebook": json.dumps(notebook)}).encode("utf-8")
        request = urllib.request.Request(address, data=body, headers={"Content-Type": "application/json"}, method="POST")
        with urllib.request.urlopen(request, timeout=60) as reply:
            return json.loads(reply.read().decode("utf-8"))
    def loop():
        while True:
            try:
                answer = send() or {}
                result = answer.get("result") or {}
                if result.get("other_file"):
                    note.update(HTML('<div style="border:3px solid #b3261e;background:#fff0ee;padding:10px;font-size:16px;color:#7a1712">'
                                     '<b>You are working in a different file from your issued notebook.</b> Your work here is still backed up. '
                                     'Go back to the entry page and click Resume, or keep working here and tell your lecturer on the class channel.</div>'))
                elif answer.get("ok"):
                    note.update(HTML('<div style="color:#526577;font-size:13px">Automatic backup sent at ' + time.strftime("%H:%M") + '. Your notebook also saves itself; press Ctrl+S now and then.</div>'))
                elif result.get("closed") or "deadline" in str(answer.get("error", "")):
                    note.update(HTML('<div style="color:#526577;font-size:13px">Your time has ended. Your work was collected at your deadline.</div>'))
                    return
            except Exception:
                pass
            time.sleep(minutes * 60)
    thread = threading.Thread(target=loop, name="iuea-backup", daemon=True)
    thread.start()
    globals()["_iuea_backup_thread"] = thread
_iuea_backup_start()

def _iuea_countdown_start():
    import datetime, uuid
    try:
        from IPython import get_ipython
        from IPython.display import display, HTML
        shell = get_ipython()
        if shell is None or "google.colab" not in type(shell).__module__:
            return
    except Exception:
        return
    if not _IUEA_DEADLINE:
        display(HTML('<div style="border:3px solid #b3261e;background:#fff0ee;padding:10px;font:16px Arial;color:#7a1712"><b>No timer: this notebook has no deadline.</b> '
                     'Open your notebook from the entry page (Open my project in Colab), not a copy.</div>'))
        return
    end = datetime.datetime.fromisoformat(_IUEA_DEADLINE.replace("Z", "+00:00"))
    local = (end + datetime.timedelta(hours=3)).strftime("%H:%M")
    practice = TEST_ID == "ML_REHEARSAL"
    def text():
        left = (end - datetime.datetime.now(datetime.timezone.utc)).total_seconds()
        if left <= 0:
            return "Time is up (deadline " + local + " Kampala)" + ("." if practice else ". Your work is collected.")
        m = int(left // 60)
        out = "%d:%02d left. Deadline %s Kampala." % (m // 60, m % 60, local)
        if m < 30:
            out += " Press Ctrl+S" + (" now: 10 minutes or less." if m < 10 else ".")
        return out
    def clock(size):
        # The text is right at once; the script keeps it ticking (it uses this device's clock, the server keeps the real deadline).
        box = "iuea" + uuid.uuid4().hex[:12]
        return HTML('<div style="font:bold ' + size + ' Arial;color:#126573;margin:4px 0">⏱ <span id="' + box + '">' + text() + '</span></div>'
                    '<script>(function(){const s=document.getElementById("' + box + '"),end=Date.parse("' + _IUEA_DEADLINE + '");if(!s)return;'
                    'function t(){const l=end-Date.now();if(l<=0){s.textContent="Time is up (deadline ' + local + ' Kampala).";return;}const m=Math.floor(l/60000);'
                    's.textContent=Math.floor(m/60)+":"+String(m%60).padStart(2,"0")+" left. Deadline ' + local + ' Kampala."+(m<30?" Press Ctrl+S"+(m<10?" now: 10 minutes or less.":"."):"");}'
                    't();setInterval(t,20000);})();</script>')
    first = [True]
    def line(*_):
        if first[0]:
            first[0] = False                          # the setup cell already shows the big clock
            return
        try:
            display(clock("15px"))
        except Exception:
            print("⏱ " + text())
    old = globals().get("_iuea_countdown")
    if old is not None:
        try:
            shell.events.unregister("post_run_cell", old)
        except Exception:
            pass
    display(clock("26px"))
    shell.events.register("post_run_cell", line)
    globals()["_iuea_countdown"] = line
_iuea_countdown_start()


## The data
Setup loads two tables. Both are real published data, organised for this project. **A few problems were added on purpose** for you to clean in Task 1.

**`houses_raw`**: 2,935 house sales in Ames, Iowa (USA), 2006 to 2010. One row is one sale. Question 1 predicts the price.

| Column | Meaning |
|---|---|
| `house_id` | Sale number |
| `neighbourhood` | Area of the city (text) |
| `building_type` | Detached, townhouse, duplex or two-family conversion (text) |
| `overall_quality`, `overall_condition` | Rating from 1 (very poor) to 10 (excellent) |
| `year_built`, `year_remodelled` | Year |
| `living_area_m2`, `basement_area_m2`, `lot_area_m2`, `garage_area_m2` | Area in square metres |
| `lot_frontage_m` | Length of the plot along the street, in metres |
| `bedrooms`, `full_bathrooms`, `garage_cars`, `fireplaces` | Counts |
| `kitchen_quality` | Excellent, good, average, fair or poor (text) |
| **`sale_price_usd`** | Price paid, in US dollars. **The target of Question 1.** |

**`machines_raw`**: 10,012 runs of a metal-cutting (milling) machine. The data are synthetic: generated to behave like real machine records. One row is one run. Question 2 warns of a failure.

| Column | Meaning |
|---|---|
| `machine_id` | Record number |
| `product_type` | Product quality: L (low), M (medium) or H (high) |
| `air_temp_k`, `process_temp_k` | Air and process temperature, in kelvin |
| `rotation_speed_rpm` | Spindle speed, in revolutions per minute |
| `torque_nm` | Turning force, in newton metres |
| `tool_wear_min` | Minutes the cutting tool has been in use |
| **`failure`** | 1 if the machine failed during the run, 0 if not. **The target of Question 2.** |

**Problems to clean (Task 1):** duplicate rows, rows with no target value, and impossible values (for example a negative area). Some values are also missing, for example `lot_frontage_m`: leave those for the imputer.

## Task 1: Clean the two tables (6 marks)
Start from a copy (`houses = houses_raw.copy()`, the same for machines), then for each table:
1. Remove duplicate rows.
2. Remove the rows with no target value (`H_TARGET` for houses, `M_TARGET` for machines).
3. Remove the rows with an impossible value. Houses: `living_area_m2` must be above 0 and `year_built` must be 2010 or earlier. Machines: `torque_nm` and `rotation_speed_rpm` must be above 0.
4. Machines only: change `product_type` to a number (L = 0, M = 1, H = 2).

Leave the other missing values: in Task 3 the imputer fills them from the training rows only.
**Chart 1:** a scatter of `living_area_m2` (across) against `H_TARGET` (up), from `houses`, with a title, axis labels and the unit (USD).

**Input:** `houses_raw`, `machines_raw`. **Store:** `houses`, `machines`.

In [ ]:
# Your code here. Store: houses, machines


## Task 2: Split into training, validation and test sets (2 marks)
Use exactly these four lines for houses, then the same four for machines:
`houses_test = houses.sample(frac=0.2, random_state=17)`  
`houses_rest = houses.drop(houses_test.index)`  
`houses_valid = houses_rest.sample(frac=0.25, random_state=17)`  
`houses_train = houses_rest.drop(houses_valid.index)`

This gives 60 percent training, 20 percent validation and 20 percent test rows. Do not use the test rows until Task 6.

**Input:** `houses`, `machines`. **Store:** `houses_train`, `houses_valid`, `houses_test`, `machines_train`, `machines_valid`, `machines_test`.

In [ ]:
# Your code here. Store: houses_train, houses_valid, houses_test, machines_train, machines_valid, machines_test


## Task 3: Question 1: predict the sale price of a house (13 marks)
Fit three models on `houses_train` (columns `H_FEATURES`, target `H_TARGET`), each as `make_pipeline(SimpleImputer(strategy="median"), model)`: `DummyRegressor()`, `LinearRegression()` and `DecisionTreeRegressor(max_depth=4, random_state=17)`.
Predict `houses_valid` and compute the MAE and RMSE of each, in USD.
**Chart 2:** a bar chart of the three MAE values, with a title, axis labels and the unit (USD).

**Input:** `houses_train`, `houses_valid`. **Store:** `baseline_pred`, `linear_pred`, `tree_pred`, `baseline_mae`, `linear_mae`, `tree_mae`, `baseline_rmse`, `linear_rmse`, `tree_rmse`.

In [ ]:
# Your code here. Store: baseline_pred, linear_pred, tree_pred, baseline_mae, linear_mae, tree_mae, baseline_rmse, linear_rmse, tree_rmse


## Task 3b: Do the text columns help? (2 marks)
A comparison only: not one of the models in Task 5. Fit this pipeline on `houses_train` (columns `H_FEATURES + TEXT_FEATURES`):
`make_pipeline(make_column_transformer((SimpleImputer(strategy="median"), H_FEATURES), (make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), TEXT_FEATURES)), LinearRegression())`
Predict `houses_valid` and compute the MAE, in USD.

**Input:** `houses_train`, `houses_valid`. **Store:** `encoded_pred`, `encoded_mae`.

In [ ]:
# Your code here. Store: encoded_pred, encoded_mae


## Task 4: Question 2: warn of a machine failure (3 marks)
A warning is a reading with `failure` at or above `M_LIMIT` (1): `M_TARGET >= M_LIMIT`. Fit `make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000))` on `machines_train` (columns `M_FEATURES`). Store the probability of a warning for each `machines_valid` row.
Warn when the probability is at least `THRESHOLD` (0.5). Compare with the real warnings in `machines_valid`; store the counts, precision and recall (0 when a count is zero).

**Input:** `machines_train`, `machines_valid`. **Store:** `warning_proba`, `tp`, `fp`, `fn`, `tn`, `precision`, `recall`.

In [ ]:
# Your code here. Store: warning_proba, tp, fp, fn, tn, precision, recall


## Task 5: Choose one model
Run `chosen_model = choose_model({"baseline": baseline_mae, "linear": linear_mae, "tree": tree_mae})`. The rule keeps models with MAE at most `MAX_MAE` (25,000 USD) and time at most `MAX_MS`, then takes the smallest MAE. Never use the test rows to choose.

**Input:** `baseline_mae`, `linear_mae`, `tree_mae`. **Store:** `chosen_model`.

In [ ]:
# Your code here. Store: chosen_model


## Task 6: Use the test set once (2 marks)
Only now, after the choice in Task 5, use the test rows, once.
1. Refit the chosen model on `houses_train` and `houses_valid` together. Store its MAE on `houses_test` as `test_mae`.
2. Refit the warning model on `machines_train` and `machines_valid` together. Store its recall on `machines_test` at 0.5 as `test_recall`.

**Input:** `houses_train`, `houses_valid`, `houses_test`, `machines_train`, `machines_valid`, `machines_test`, `chosen_model`. **Store:** `test_mae`, `test_recall`.

In [ ]:
# Your code here. Store: test_mae, test_recall


## Task 7: Part B, the decision memo form (15 automatic marks)
Fill in `MEMO` in the next cell, for a property valuer (and, for the warning, the maintenance manager). Use your own numbers, with at least two decimals, in USD.

| Field | Enter | Marks |
|---|---|---|
| `recommended_model` | `"baseline"`, `"linear"` or `"tree"`: the model the rule chose and you opened the test set with. | 3 |
| `validation_mae` | The validation MAE of each model, from your notebook. | 3 |
| `test_mae`, `test_recall` | The MAE and the warning recall (0 to 1) from your one test opening. | 2 |
| `improvement_over_baseline` | Baseline validation MAE minus your model's (0 if you recommend the baseline). | 1 |
| `limits` | Three different letters: the true limits of this study. One mark each. | 3 |
| `next_step` | One letter: the best next step. | 1 |
| `answer_choice`, `answer_test_set` | Two answers in your own words, 8 to 80 words each. One mark each. | 2 |

**Limits** (some statements are false):
- **A.** Fitting the imputer on the training rows only makes the validation results unreliable.
- **B.** Because the test rows were used only once, the test results have no uncertainty at all.
- **C.** The test rows are one random fifth of each table, so the test results are one estimate that could be luckier or unluckier than the truth.
- **D.** The client limits (the error limit and the time limit) are invented teaching numbers; a real valuer would set their own.
- **E.** The test MAE proves that the model will be this accurate for every house in every town.
- **F.** The machine data are synthetic (generated to look like real maintenance data), so the warning results may not hold on a real machine.
- **G.** The model with the lowest training error is always the best model for new houses.
- **H.** The baseline model should be left out of the comparison because it uses no features.
- **I.** All the house sales come from one city (Ames, Iowa) between 2006 and 2010, so the model may not hold for other towns or later years.
- **J.** RMSE is always smaller than MAE, so MAE is the more severe measure and should be the only one reported.

**Next step:**
- **A.** Move the test rows into the training rows so the model learns from more houses, and report the training error.
- **B.** Remove the baseline model from the report so the results look better.
- **C.** Collect sales from more towns and years, and real machine records, then check the chosen models again on rows they have never seen before any decision.
- **D.** Use the chosen model for every valuation now, because the test MAE is below the client limit.
- **E.** Tune the maximum depth of the decision tree until its test MAE is lowest, then report that test MAE.

**Written answers:**
- `answer_choice`: In one or two sentences: why do you recommend this model? Use a number from the validation set and say what you compared it with.
- `answer_test_set`: In one or two sentences: why is the test set used only once, at the very end, and never used to choose the model?

In [ ]:
# Fill in every field; do not rename a field.
MEMO = {
    "recommended_model": "",                  # "baseline", "linear" or "tree"
    "validation_mae": {"baseline": None, "linear": None, "tree": None},   # your validation MAE values, in USD
    "test_mae": None,                         # test_mae from Task 6, in USD
    "test_recall": None,                      # test_recall from Task 6 (0 to 1)
    "improvement_over_baseline": None,        # baseline validation MAE minus your recommended model's, in USD
    "limits": ["", "", ""],                   # three different letters from the Limits list
    "next_step": "",                          # one letter from the Next step list
    "answer_choice": "",                      # one or two sentences
    "answer_test_set": "",                    # one or two sentences
}
